In [38]:
import pandas as pd
from pycaret.regression import setup, compare_models, predict_model, save_model, load_model
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LinearRegression
from pycaret.classification import *
from sklearn.metrics import log_loss, f1_score

In [29]:
df_prod = pd.read_parquet("../data/04_feature/data_filtered.parquet")
df_prod

,lat,lon,minutes_remaining,period,playoffs,shot_distance,shot_made_flag
1,34.0443,-118.4268,10,1,0,15,0.0
2,33.9093,-118.3708,7,1,0,16,1.0
3,33.8693,-118.1318,6,1,0,22,0.0
4,34.0443,-118.2698,6,2,0,0,1.0
5,34.0553,-118.4148,9,3,0,14,0.0
...,...,...,...,...,...,...,...
30690,33.9443,-118.3828,11,4,1,15,0.0
30691,34.0443,-118.2698,7,4,1,0,0.0
30692,33.9963,-118.2688,6,4,1,4,0.0
30694,33.8783,-118.4038,3,4,1,21,1.0


In [14]:
features = df_prod.drop("shot_made_flag", axis=1)
target = df_prod['shot_made_flag']

In [4]:
exp_clf = setup(df_prod, target='shot_made_flag')


,Description,Value
0,Session id,3543
1,Target,shot_made_flag
2,Target type,Regression
3,Original data shape,"(20285, 7)"
4,Transformed data shape,"(20285, 7)"
5,Transformed train set shape,"(14199, 7)"
6,Transformed test set shape,"(6086, 7)"
7,Numeric features,6
8,Preprocess,True
9,Imputation type,simple


In [5]:
best = compare_models(sort='MAE')

<IPython.core.display.HTML object>

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
knn,K Neighbors Regressor,0.4761,0.2851,0.5339,-0.1430,0.3733,0.4936,0.0100
et,Extra Trees Regressor,0.4774,0.2781,0.5274,-0.1151,0.3691,0.4936,0.1030
xgboost,Extreme Gradient Boosting,0.4786,0.2630,0.5128,-0.0545,0.3589,0.4988,0.0690
lightgbm,Light Gradient Boosting Machine,0.4790,0.2450,0.4949,0.0179,0.3476,0.4993,252.4660
catboost,CatBoost Regressor,0.4791,0.2472,0.4972,0.0088,0.3491,0.4993,2.0230
rf,Random Forest Regressor,0.4800,0.2634,0.5132,-0.0559,0.3598,0.4956,0.1730
gbr,Gradient Boosting Regressor,0.4801,0.2414,0.4913,0.0323,0.3453,0.4999,0.0740
ada,AdaBoost Regressor,0.4807,0.2416,0.4915,0.0313,0.3455,0.4996,0.0070
lar,Least Angle Regression,0.4846,0.2424,0.4924,0.0280,0.3462,0.5046,0.0060
lr,Linear Regression,0.4846,0.2424,0.4924,0.0280,0.3462,0.5046,0.2740


<IPython.core.display.HTML object>

In [15]:
prediction_df = predict_model(best, features)

<IPython.core.display.HTML object>

In [16]:
save_model(best, 'knn')

Transformation Pipeline and Model Successfully Saved



(
    Pipeline(memory=Memory(location=None),
         steps=[('numerical_imputer',
                 TransformerWrapper(include=['lat', 'lon', 'minutes_remaining',
                                             'period', 'playoffs',
                                             'shot_distance'],
                                    transformer=SimpleImputer())),
                ('categorical_imputer',
                 TransformerWrapper(include=[],
                                    transformer=SimpleImputer(strategy='most_frequent'))),
                ('trained_model', KNeighborsRegressor(n_jobs=-1))]),
    'knn.pkl'
)

In [41]:
knn_model = load_model('knn')

Transformation Pipeline and Model Successfully Loaded


In [42]:
knn_tuned = tune_model(knn_model)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 knn_tuned = tune_model(knn_model)                                                            │
│   2                                                                                              │
│                                                                                                  │
│ /home/bakudas/code/INFNET_POS/pos-infnet-eng-machine-learning/.venv/lib/python3.11/site-packages │
│ /pycaret/utils/generic.py:969 in wrapper                                                         │
│                                                                                                  │
│    966 │   │   def wrapper(*args, **kwargs):                                                     │
│    967 │   │   │   for name, message in global_names.items():                                    │
│    968 │   │   │   │   if globals_d[name] is None:                                               │
│ ❱  969 │   │   │   │   │   raise ValueError(message)                                             │
│    970 │   │   │   return func(*args, **kwargs)                                                  │
│    971 │   │                                                                                     │
│    972 │   │   return wrapper                                                                    │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ValueError: _CURRENT_EXPERIMENT global variable is not set. Please run setup() first.

In [30]:
relevant_cols = ["lat", "lon", "minutes_remaining", "period", "playoffs", "shot_distance"]

df_prod_filtered = df_prod.dropna(subset=relevant_cols)

# Manter somente as colunas relevantes
df_prod_filtered = df_prod_filtered[relevant_cols + ["shot_made_flag"] 
                                    if "shot_made_flag" in df_prod.columns 
                                    else relevant_cols].dropna()

print("Dimensão do df_prod_filtered:", df_prod_filtered.shape)
df_prod_filtered.head(3)

predictions = predict_model(knn_model, data=df_prod)

Dimensão do df_prod_filtered: (20285, 7)


In [31]:
# ver as colunas e algumas linhas
print("Colunas do DataFrame de predições:", predictions.columns)
predictions.head(3)

Colunas do DataFrame de predições: Index(['lat', 'lon', 'minutes_remaining', 'period', 'playoffs',
       'shot_distance', 'shot_made_flag', 'prediction_label'],
      dtype='object')


,lat,lon,minutes_remaining,period,playoffs,shot_distance,shot_made_flag,prediction_label
1,34.044300,-118.426804,10,1,0,15,0.0,0
2,33.909302,-118.370796,7,1,0,16,1.0,0
3,33.869301,-118.131798,6,1,0,22,0.0,0


In [35]:
if "shot_made_flag" in df_prod_filtered.columns:
    # Descubra se a probabilidade está em "Score" ou "Score_1"
    # Caso você não use raw_score=True, normalmente "Score" é a probabilidade da classe positiva
    y_true = predictions["shot_made_flag"]
    y_pred = predictions["prediction_label"]
    #y_proba = predictions["prediction_score"]

    # Calcular métricas
    ll_prod = log_loss(y_true, y_pred)
    f1_prod = f1_score(y_true, y_pred)

    print(f"Log Loss na base de produção: {ll_prod:.4f}")
    print(f"F1 Score na base de produção: {f1_prod:.4f}")
else:
    print("Não há 'shot_made_flag' na base de produção, não é possível calcular métricas supervisionadas.")

Log Loss na base de produção: 16.3702
F1 Score na base de produção: 0.1252
